# 04 — RGVH Filter Strategy

Always-short SPY ATM straddle, gated by three independent regime filters.

This notebook reproduces the headline result of Sharpe 3.38 on the 12.2-year OOS window. **Thresholds in `config.py` are NaN by default** — set your own values after re-running the threshold sensitivity sweep on your data.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src import backtest, filters, evaluate
from src.config import IV_THR, VXN_THR, SLOPE_THR
print('thresholds:', IV_THR, VXN_THR, SLOPE_THR)
# Edit src/config.py and rerun if these are NaN.

## 1. Run the always-short baseline

In [ ]:
trades = backtest.run_baseline_short_vol(
    opts_path='../data/processed/spy_eod_unified.parquet',
    panel_path='../data/processed/factor_panel_daily.parquet',
    output_path='../data/processed/baseline_trades.parquet',
)
trades.head(3)

## 2. Merge in filter features

In [ ]:
panel = pd.read_parquet('../data/processed/factor_panel_daily.parquet')
panel['tradeDate'] = pd.to_datetime(panel['tradeDate'])

trades['entry_date'] = pd.to_datetime(trades['entry_date'])
trades = trades.merge(
    panel[['tradeDate','F_iv_rank_252','vxn_excess_rank_252','slope_2s10s_rank_252']]
         .rename(columns={'tradeDate':'entry_date','F_iv_rank_252':'iv_rank'}),
    on='entry_date', how='left'
)
trades.head(3)

## 3. Apply the three-filter skip mask

In [ ]:
skip = filters.build_skip_mask(trades)
rgvh = trades.loc[~skip].reset_index(drop=True)
print(f'kept {len(rgvh)} of {len(trades)} ({len(rgvh)/len(trades):.1%})')

## 4. Compare baseline vs RGVH

In [ ]:
summary = pd.DataFrame([
    evaluate.summary(trades,  'always-short baseline'),
    evaluate.summary(rgvh,    'RGVH'),
])
summary

## 5. Equity curve

In [ ]:
from src.evaluate import daily_pnl_series
fig, ax = plt.subplots(figsize=(11,4))
daily_pnl_series(trades).cumsum().plot(ax=ax, label='Always short', color='grey', ls='--')
daily_pnl_series(rgvh).cumsum().plot(ax=ax, label='RGVH', color='green', lw=2)
ax.legend(); ax.set_title('Cumulative net P&L'); ax.axhline(0, color='black', lw=0.5)

## 6. Year-by-year

In [ ]:
evaluate.yearly_breakdown(rgvh)